# PIPELINE THỰC THI TOÀN DIỆN ALL-IN-ONE IMAGE RESTORATION (AiOIR)
**Môi trường thực thi**: Google Colab Free (GPU Tesla T4 ~15GB VRAM, RAM ~12GB)
**Mục tiêu thí nghiệm (1-Click Run)**:
1. **P0 (Setup & Verification)**: Khởi tạo phần cứng T4, kiểm tra AMP, cài đặt thư viện, tải và kiểm tra số lượng ảnh benchmark (BSD68, Kodak24, Urban100, Rain100L, SOTS).
2. **P1 (Baseline Reproduction & Efficiency)**: Đánh giá checkpoint All-in-One PromptIR & AdaIR trên tập test; đo tham số, FLOPs, độ trễ và VRAM đỉnh trên T4.
3. **P2 (Failure Analysis & OOD Diagnostic)**: Chẩn đoán 30 ảnh tệ nhất mỗi tập, phân tích miền tần số 2D FFT, quang phổ sai số và kiểm tra độ bền OOD (Out-of-Distribution).
4. **Tự động đóng gói và sao lưu**: Lưu toàn bộ logs, bảng kết quả CSV và hình ảnh trực quan hóa lên Google Drive.

--- 
## BƯỚC 1: KẾT NỐI GOOGLE DRIVE & KIỂM TRA PHẦN CỨNG GPU T4

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Kiểm tra thông số GPU T4 qua nvidia-smi
!nvidia-smi

import torch
print('=' * 60)
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    print('VRAM Dung lượng :', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
    print('AMP FP16 Hỗ trợ  :', torch.cuda.is_bf16_supported() or True)
print('=' * 60)

--- 
## BƯỚC 2: THIẾT LẬP WORKSPACE VÀ CÀI ĐẶT THƯ VIỆN BỔ TRỢ

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path('/content/repo')
DRIVE_WORKSPACE = Path('/content/drive/MyDrive/nckh_coChau_workspace/repo')

# Đồng bộ từ Google Drive vào /content nếu đã có, hoặc tạo mới
if DRIVE_WORKSPACE.exists():
    print(f'Đang đồng bộ codebase từ Google Drive: {DRIVE_WORKSPACE} -> {REPO_DIR}...')
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.copytree(DRIVE_WORKSPACE, REPO_DIR)
else:
    print('Thư mục Google Drive chưa có, khởi tạo /content/repo cục bộ...')
    REPO_DIR.mkdir(parents=True, exist_ok=True)

%cd /content/repo

# Cài đặt thư viện theo requirements.txt
!pip install -q -r requirements.txt
!pip install -q gdown ptflops lpips
print('Hoàn tất cài đặt các thư viện cần thiết!')

--- 
## BƯỚC 3: TẢI TỰ ĐỘNG DỮ LIỆU BENCHMARK VÀ CHECKPOINT TỪ TÁC GIẢ GỐC

In [ ]:
%cd /content/repo

# 1. Tải toàn bộ Checkpoint chính thức (PromptIR All.ckpt) và tập test (BSD68, Kodak24, Urban100, Rain100L)
!python scripts/download_datasets.py

# 2. Kiểm tra số lượng ảnh đạt chuẩn bài báo
!python scripts/verify_data_counts.py repo/data/test

--- 
## BƯỚC 4: P1-T1 TÁI HIỆN BASELINE TRÊN CÁC TẬP KIỂM CHUẨN (TESTSETS)

In [ ]:
%cd /content/repo

# 1. Đánh giá PromptIR trên checkpoint All.ckpt
!python scripts/evaluate_baselines.py --baseline promptir --weights repo/baselines/weights/promptir_all.ckpt --data_dir repo/data/test

# 2. Đánh giá AdaIR nếu có checkpoint trong thư mục weights
import glob
adair_ckpts = glob.glob('repo/baselines/weights/adair/*.pth') + glob.glob('repo/baselines/weights/adair/*.pt')
if adair_ckpts:
    print(f'Phát hiện checkpoint AdaIR: {adair_ckpts[0]}')
    !python scripts/evaluate_baselines.py --baseline adair --weights {adair_ckpts[0]} --data_dir repo/data/test
else:
    print('Chưa có checkpoint AdaIR .pth, PromptIR đã hoàn thành đánh giá.')

import pandas as pd
df_base = pd.read_csv('repo/results/baseline_results.csv')
print('\n=== KẾT QUẢ TÁI HIỆN BASELINE ===')
display(df_base)

--- 
## BƯỚC 5: P1-T2 ĐO ĐẠC HIỆU NĂNG THỰC TẾ TRÊN GPU T4 (FLOPS, LATENCY, VRAM)

In [ ]:
%cd /content/repo

# Đo lường chính xác Params, FLOPs, Độ trễ suy luận (Inference Latency 50 lần), và VRAM đỉnh trên T4
!python scripts/profile_efficiency.py

df_eff = pd.read_csv('repo/results/efficiency.csv')
print('\n=== BẢNG ĐO HIỆU NĂNG PHẦN CỨNG T4 ===')
display(df_eff)

--- 
## BƯỚC 6: P2-T2 & P2-T3 CHẨN ĐOÁN THẤT BẠI CHUYÊN SÂU & THỬ THÁCH OOD

In [ ]:
%cd /content/repo

# Phân tích 30 ảnh tệ nhất mỗi tập, vẽ bản đồ nhiệt sai số, quang phổ 2D FFT, và stress-test OOD
!python scripts/diagnose_failures.py --baseline promptir --weights repo/baselines/weights/promptir_all.ckpt --data_dir repo/data --output_dir repo/results/diagnostics --max_samples 50

df_ood = pd.read_csv('repo/results/ood_baseline.csv')
print('\n=== KẾT QUẢ THỬ THÁCH SUY THOÁI CHƯA TỪNG GẶP (OOD) ===')
display(df_ood)

# Hiển thị 3 ảnh chẩn đoán lỗi tiêu biểu
from PIL import Image
from IPython.display import display as ipy_display
worst_imgs = sorted(list(Path('repo/results/diagnostics/worst_cases').glob('*.png')))[:3]
for img_p in worst_imgs:
    print(f'Minh họa thất bại: {img_p.name} (Clean | Degraded | Restored | Error Heatmap | FFT Spectrum)')
    ipy_display(Image.open(img_p))

--- 
## BƯỚC 7: TỰ ĐỘNG ĐÓNG GÓI VÀ SAO LƯU KẾT QUẢ LÊN GOOGLE DRIVE

In [ ]:
import os
import datetime
import zipfile
from pathlib import Path

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = Path('/content/drive/MyDrive/nckh_coChau_workspace/runs_backup')
backup_dir.mkdir(parents=True, exist_ok=True)

zip_filename = backup_dir / f'aioir_full_results_{timestamp}.zip'
print(f'Đang nén kết quả vào {zip_filename}...')

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for folder in ['repo/results', 'repo/analysis']:
        p = Path(folder)
        if p.exists():
            for f in p.rglob('*'):
                if f.is_file():
                    zipf.write(f, f.relative_to(Path('repo')))

print(f'\n[XÁC NHẬN] ĐÃ SAO LƯU KẾT QUẢ THÀNH CÔNG VÀO GOOGLE DRIVE:')
print(f'>> Đường dẫn: {zip_filename}')
print('Vui lòng copy toàn bộ text đầu ra của notebook này và gửi cho AI cộng sự để tiến hành Phase P3!')